# Chronos 2 Foundation Model Forecasting

This notebook implements zero-shot forecasting for hourly bike rental demand using Chronos 2.

The following Chronos 2 strategies are evaluated:
- Univariate Chronos 2,
- Covariate Chronos 2.

The objective of this notebook is to validate both strategies using an expanding-window approach, select the best setup based on validation MASE, and finally evaluate the selected model on the test set.

Databricks execution note: run this notebook on a GPU-enabled cluster with AutoGluon TimeSeries installed. If AutoGluon or Chronos is missing, install it first with `%pip install -U "autogluon.timeseries[chronos]" "chronos-forecasting>=2.0"`, restart Python, and then rerun the notebook.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [ ]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "processed" / "train.csv").exists():
            return candidate_path

    raise FileNotFoundError(
        "Could not find project root. Run this notebook from the repository "
        "or make sure data/processed/train.csv exists."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FORECAST_DIR = PROJECT_ROOT / "outputs" / "forecasts"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

os.makedirs(FORECAST_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

TARGET = "cnt"
DATE_COLUMN = "dteday"
HOUR_COLUMN = "hr"
TIMESTAMP_COLUMN = "timestamp"

ID_COLUMN = "item_id"
ITEM_ID = "bike_rentals"

FREQUENCY = "h"
FORECAST_HORIZON = 24
VALIDATION_STEP = FORECAST_HORIZON
VALIDATION_METRIC = "MASE"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load Chronological Data Splits

The train, validation and test sets were created previously during the preprocessing stage. The train set is used as historical context for zero-shot forecasting, the validation set is used only for setup selection, and the test set stays untouched until the final evaluation.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val = pd.read_csv(f"{DATA_DIR}/val.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")


def add_hourly_timestamp(df):
    df = df.copy()

    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df[TIMESTAMP_COLUMN] = (
        df[DATE_COLUMN] + pd.to_timedelta(df[HOUR_COLUMN].astype(int), unit="h")
    )

    df = df.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

    if df[TIMESTAMP_COLUMN].duplicated().any():
        duplicated_timestamps = df.loc[
            df[TIMESTAMP_COLUMN].duplicated(), TIMESTAMP_COLUMN
        ].head()

        raise ValueError(
            f"Duplicate hourly timestamps found: {duplicated_timestamps.tolist()}"
        )

    return df


train = add_hourly_timestamp(train)
val = add_hourly_timestamp(val)
test = add_hourly_timestamp(test)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

print("Duplicate timestamps:")
print("train:", train[TIMESTAMP_COLUMN].duplicated().sum())
print("val:", val[TIMESTAMP_COLUMN].duplicated().sum())
print("test:", test[TIMESTAMP_COLUMN].duplicated().sum())

In [ ]:
required_columns = [TARGET]

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = set(required_columns) - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )

In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "start": [
        train[TIMESTAMP_COLUMN].min(),
        val[TIMESTAMP_COLUMN].min(),
        test[TIMESTAMP_COLUMN].min()
    ],
    "end": [
        train[TIMESTAMP_COLUMN].max(),
        val[TIMESTAMP_COLUMN].max(),
        test[TIMESTAMP_COLUMN].max()
    ],
    "target_mean": [
        train[TARGET].mean(),
        val[TARGET].mean(),
        test[TARGET].mean()
    ]
})

split_summary


In [ ]:
train.head()

# Prepare Data For AutoGluon Chronos 2

Chronos 2 will be used through AutoGluon TimeSeries. In this step, we prepare two clean modelling datasets: one only with the target variable, and one with target plus selected calendar and weather covariates.

In [ ]:
try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
except ImportError as error:
    raise ImportError(
        "AutoGluon TimeSeries and Chronos are required to run this notebook. "
        "Install them before running Chronos 2, for example with: "
        "%pip install -U \"autogluon.timeseries[chronos]\" \"chronos-forecasting>=2.0\""
    ) from error


# Select Modelling Columns

For the univariate setup we use only `cnt`. For the covariate setup we add calendar variables and weather variables, treating weather information as future available predictions for this project.

In [ ]:
ID_COLUMN = "item_id"
TIMESTAMP_COLUMN = "timestamp"

CALENDAR_COVARIATES = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday"
]

WEATHER_COVARIATES = [
    "weathersit",
    "temp",
    "hum",
    "windspeed"
]

KNOWN_COVARIATES = CALENDAR_COVARIATES + WEATHER_COVARIATES

UNIVARIATE_COLUMNS = [TARGET]
COVARIATE_COLUMNS = [TARGET] + KNOWN_COVARIATES

EXCLUDED_ENGINEERED_FEATURES = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168"
]

In [ ]:
selected_columns = pd.DataFrame({
    "column_group": (
        ["target"]
        + ["calendar_covariate"] * len(CALENDAR_COVARIATES)
        + ["weather_covariate"] * len(WEATHER_COVARIATES)
    ),
    "column_name": [TARGET] + KNOWN_COVARIATES
})

selected_columns

In [ ]:
required_columns = set(COVARIATE_COLUMNS + [TIMESTAMP_COLUMN])

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = required_columns - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )


# Convert Data To AutoGluon Format

AutoGluon TimeSeries needs a long dataframe with an item identifier and timestamp. Because we have one time series, the `item_id` is constant for all observations. We create AutoGluon training objects only for the initial predictor setup; validation and test histories are converted inside the expanding-window function.

In [ ]:
def prepare_autogluon_dataframe(df, modelling_columns):
    model_df = df[[TIMESTAMP_COLUMN] + modelling_columns].copy()
    model_df[ID_COLUMN] = ITEM_ID

    return model_df[[ID_COLUMN, TIMESTAMP_COLUMN] + modelling_columns]


def to_timeseries_dataframe(df):
    return TimeSeriesDataFrame.from_data_frame(
        df,
        id_column=ID_COLUMN,
        timestamp_column=TIMESTAMP_COLUMN
    )

In [ ]:
train_univariate_df = prepare_autogluon_dataframe(
    train,
    UNIVARIATE_COLUMNS
)

train_covariate_df = prepare_autogluon_dataframe(
    train,
    COVARIATE_COLUMNS
)

train_univariate_data = to_timeseries_dataframe(
    train_univariate_df
)

train_covariate_data = to_timeseries_dataframe(
    train_covariate_df
)

In [ ]:
print("Univariate training data shape:", train_univariate_data.shape)
print("Covariate training data shape:", train_covariate_data.shape)

train_covariate_df.head() 

# Evaluation Metrics

We use MASE as the main validation metric for choosing the better Chronos 2 setup. MAE and RMSE are also calculated, because they make the results easier to compare with the other models in the project. MASE is calculated with a seasonal naive benchmark using a 24-hour seasonal period.


In [ ]:
POINT_FORECAST_COLUMN = "mean"


def rmse(
    y_true,
    y_pred
):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def mase(
    y_true,
    y_pred,
    train_series,
    seasonal_period=24
):
    seasonal_naive_error = np.mean(
        np.abs(
            train_series[seasonal_period:].values
            - train_series[:-seasonal_period].values
        )
    )

    if seasonal_naive_error == 0:
        raise ValueError(
            "MASE cannot be calculated because the seasonal naive forecast error is zero."
        )

    model_mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return model_mae / seasonal_naive_error


def evaluate_forecasts(
    y_true,
    y_pred,
    train_series
):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "RMSE": rmse(
            y_true,
            y_pred
        ),
        "MASE": mase(
            y_true,
            y_pred,
            train_series,
            seasonal_period=FORECAST_HORIZON
        )
    }


# Expanding-Window Validation Logic

We follow the same expanding-window validation idea as in the LightGBM notebook. The model forecasts the next 24 hours, then the real observed validation values are added to history, and the next 24-hour block is forecasted. For the covariate setup, future covariates are prepared for the full forecast horizon, as required by AutoGluon.

In [ ]:
def get_forecast_starts(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    return range(
        0,
        len(evaluation_df) - forecast_horizon,
        step_size
    )


def summarize_expanding_window_setup(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    forecast_starts = list(
        get_forecast_starts(
            evaluation_df,
            forecast_horizon,
            step_size
        )
    )

    evaluated_rows = len(forecast_starts) * forecast_horizon
    non_evaluated_rows = len(evaluation_df) - evaluated_rows

    return pd.DataFrame({
        "forecast_horizon": [forecast_horizon],
        "step_size": [step_size],
        "forecast_windows": [len(forecast_starts)],
        "available_rows": [len(evaluation_df)],
        "evaluated_rows": [evaluated_rows],
        "non_evaluated_rows": [non_evaluated_rows]
    })

In [ ]:
validation_window_summary = summarize_expanding_window_setup(
    val,
    FORECAST_HORIZON,
    VALIDATION_STEP
)

test_window_summary = summarize_expanding_window_setup(
    test,
    FORECAST_HORIZON
)

pd.concat(
    [
        validation_window_summary.assign(split="validation"),
        test_window_summary.assign(split="test")
    ],
    ignore_index=True
)[[
    "split",
    "forecast_horizon",
    "step_size",
    "forecast_windows",
    "available_rows",
    "evaluated_rows",
    "non_evaluated_rows"
]]

In [ ]:
def extract_point_forecast(
    predictions,
    point_forecast_column=POINT_FORECAST_COLUMN
):
    if point_forecast_column not in predictions.columns:
        raise ValueError(
            f"Column '{point_forecast_column}' is not available in predictions. "
            f"Available columns: {list(predictions.columns)}"
        )

    point_forecast = predictions[point_forecast_column]

    if isinstance(point_forecast.index, pd.MultiIndex):
        try:
            point_forecast = point_forecast.xs(
                ITEM_ID,
                level=ID_COLUMN
            )
        except KeyError:
            point_forecast = point_forecast.xs(
                ITEM_ID,
                level=0
            )

    return point_forecast

In [ ]:
def create_future_known_covariates(
    evaluation_df,
    start,
    known_covariates=KNOWN_COVARIATES,
    forecast_horizon=FORECAST_HORIZON
):
    future_df = evaluation_df.iloc[start:start + forecast_horizon].copy()
    future_df = future_df[[TIMESTAMP_COLUMN] + known_covariates].copy()
    future_df[ID_COLUMN] = ITEM_ID

    return TimeSeriesDataFrame.from_data_frame(
        future_df,
        id_column=ID_COLUMN,
        timestamp_column=TIMESTAMP_COLUMN
    )


def run_expanding_window_forecast(
    predictor,
    context_df,
    evaluation_df,
    modelling_columns,
    use_covariates=False,
    known_covariates=KNOWN_COVARIATES,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON,
    point_forecast_column=POINT_FORECAST_COLUMN
):
    all_results = []

    forecast_starts = get_forecast_starts(
        evaluation_df,
        forecast_horizon,
        step_size
    )

    for start in forecast_starts:
        history = pd.concat([
            context_df,
            evaluation_df.iloc[:start]
        ]).copy()

        history_data = to_timeseries_dataframe(
            prepare_autogluon_dataframe(
                history,
                modelling_columns
            )
        )

        future_known_covariates = None

        if use_covariates:
            future_known_covariates = create_future_known_covariates(
                evaluation_df=evaluation_df,
                start=start,
                known_covariates=known_covariates,
                forecast_horizon=forecast_horizon
            )

        predictions = predictor.predict(
            history_data,
            known_covariates=future_known_covariates,
            random_seed=RANDOM_SEED
        )

        point_predictions = extract_point_forecast(
            predictions,
            point_forecast_column
        )

        actual_future = evaluation_df.iloc[
            start:start + forecast_horizon
        ][[TIMESTAMP_COLUMN, TARGET]].copy()

        n_forecasts = min(
            len(actual_future),
            len(point_predictions)
        )

        actual_future = actual_future.iloc[:n_forecasts].copy()
        aligned_predictions = point_predictions.iloc[:n_forecasts]

        window_results = pd.DataFrame({
            TIMESTAMP_COLUMN: actual_future[TIMESTAMP_COLUMN].values,
            "window_start": actual_future[TIMESTAMP_COLUMN].iloc[0],
            "actual": actual_future[TARGET].values,
            "prediction": aligned_predictions.values
        })

        all_results.append(window_results)

    if not all_results:
        return pd.DataFrame(
            columns=[TIMESTAMP_COLUMN, "window_start", "actual", "prediction"]
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )


In [ ]:
def evaluate_expanding_window_results(
    forecast_df,
    train_series
):
    return evaluate_forecasts(
        forecast_df["actual"],
        forecast_df["prediction"],
        train_series
    )

# Zero-Shot Univariate Chronos 2

The first setup uses only the historical values of the target variable `cnt`. We use the standard Chronos 2 preset in zero-shot mode, so the model is not fine-tuned on our dataset. The train data is passed to `.fit()` because AutoGluon needs it to infer time-series metadata and save the predictor state, not because Chronos 2 learns new weights here.

In [ ]:
CHRONOS_PRESET = "chronos2"

MODEL_DIR = PROJECT_ROOT / "outputs" / "models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
univariate_predictor = TimeSeriesPredictor(
    prediction_length=FORECAST_HORIZON,
    target=TARGET,
    eval_metric=VALIDATION_METRIC,
    freq=FREQUENCY,
    path=str(MODEL_DIR / "chronos2_univariate"),
    verbosity=2
).fit(
    train_univariate_data,
    presets=CHRONOS_PRESET,
    enable_ensemble=False,
    random_seed=RANDOM_SEED
)

# Validate Univariate Chronos 2

We validate the univariate setup with the same expanding-window procedure used for the LightGBM model. In every window, the model receives all history available up to that point and forecasts the next 24 hours.

In [ ]:
univariate_val_forecasts = run_expanding_window_forecast(
    predictor=univariate_predictor,
    context_df=train,
    evaluation_df=val,
    modelling_columns=UNIVARIATE_COLUMNS,
    use_covariates=False,
    step_size=VALIDATION_STEP
)

univariate_val_metrics = evaluate_expanding_window_results(
    univariate_val_forecasts,
    train[TARGET]
)

univariate_val_metrics

# Zero-Shot Covariate Chronos 2

The second setup uses `cnt` together with calendar and weather covariates. Calendar variables are known in advance, and for this project we treat weather variables as future available predictions. This lets us check whether Chronos 2 benefits from additional explanatory information.

In [ ]:
covariate_predictor = TimeSeriesPredictor(
    prediction_length=FORECAST_HORIZON,
    target=TARGET,
    known_covariates_names=KNOWN_COVARIATES,
    eval_metric=VALIDATION_METRIC,
    freq=FREQUENCY,
    path=str(MODEL_DIR / "chronos2_covariate"),
    verbosity=2
).fit(
    train_covariate_data,
    presets=CHRONOS_PRESET,
    enable_ensemble=False,
    random_seed=RANDOM_SEED
)

# Validate Covariate Chronos 2

The covariate setup is validated with exactly the same expanding-window starts as the univariate setup. The only difference is that for each 24-hour forecast horizon, we also pass the known future covariates to AutoGluon.

In [ ]:
covariate_val_forecasts = run_expanding_window_forecast(
    predictor=covariate_predictor,
    context_df=train,
    evaluation_df=val,
    modelling_columns=COVARIATE_COLUMNS,
    use_covariates=True,
    step_size=VALIDATION_STEP
)

covariate_val_metrics = evaluate_expanding_window_results(
    covariate_val_forecasts,
    train[TARGET]
)

covariate_val_metrics

# Choose The Best Chronos Setup

The univariate and covariate Chronos 2 setups are compared on the validation set. The final setup is selected using validation MASE, because this is the main metric chosen for model selection in this part of the project.

In [ ]:
validation_results_df = pd.DataFrame([
    {
        "setup": "univariate_chronos2",
        "uses_covariates": False,
        **univariate_val_metrics
    },
    {
        "setup": "covariate_chronos2",
        "uses_covariates": True,
        **covariate_val_metrics
    }
])

validation_results_df = validation_results_df.sort_values(
    VALIDATION_METRIC
).reset_index(drop=True)

best_setup = validation_results_df.loc[0, "setup"]

validation_results_df["selected"] = (
    validation_results_df["setup"] == best_setup
)

validation_results_df

In [ ]:
if best_setup == "univariate_chronos2":
    best_predictor = univariate_predictor
    best_modelling_columns = UNIVARIATE_COLUMNS
    best_use_covariates = False
elif best_setup == "covariate_chronos2":
    best_predictor = covariate_predictor
    best_modelling_columns = COVARIATE_COLUMNS
    best_use_covariates = True
else:
    raise ValueError(
        f"Unknown Chronos setup: {best_setup}"
    )

print("Selected setup:", best_setup)

# Final Test Evaluation

The selected Chronos 2 setup is evaluated on the test set only once. At this stage, the available history is `train + validation`, and the test set is forecasted in the same 24-hour expanding-window blocks.

In [ ]:
final_context = pd.concat([
    train,
    val
])

chronos2_test_forecasts = run_expanding_window_forecast(
    predictor=best_predictor,
    context_df=final_context,
    evaluation_df=test,
    modelling_columns=best_modelling_columns,
    use_covariates=best_use_covariates,
    step_size=FORECAST_HORIZON
)

chronos2_test_metrics = evaluate_expanding_window_results(
    chronos2_test_forecasts,
    train[TARGET]
)

chronos2_test_metrics

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    chronos2_test_forecasts["actual"].values[:168],
    label="Actual"
)

plt.plot(
    chronos2_test_forecasts["prediction"].values[:168],
    label="Chronos 2 Forecast"
)

plt.title("Chronos 2 Test Forecasts (First 168 Hours)")
plt.xlabel("Forecast Horizon")
plt.ylabel("Bike Rentals")
plt.legend()

plt.show()

# Save Metrics And Results

The validation comparison, final test metrics and final test forecasts are saved to the project output folders. The forecast file keeps the same simple structure as the other model outputs: actual values and predictions.

In [ ]:
validation_results_df.to_csv(
    METRICS_DIR / "chronos2_validation_results.csv",
    index=False
)

chronos2_test_metrics_df = pd.DataFrame({
    "metric": list(chronos2_test_metrics.keys()),
    "value": list(chronos2_test_metrics.values())
})

chronos2_test_metrics_df.to_csv(
    METRICS_DIR / "chronos2_test_metrics.csv",
    index=False
)

chronos2_test_forecasts[["actual", "prediction"]].to_csv(
    FORECAST_DIR / "chronos2_test_forecasts.csv",
    index=False
)

chronos2_test_metrics_df 